In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

root = '/Users/samsonkoelle/isometry-pursuit'

with open(f'{root}/results/ethanol_results.json') as f:
    results = json.load(f)

with open(f'{root}/data/ethanol_metadata.json') as f:
    meta = json.load(f)

replicates_df = pd.DataFrame(results['replicates'])
selected_points = np.array(meta['selected_points'])
angles = np.load(f'{root}/{meta["angles_file"]}')

replicates_df.head()

In [ ]:
replicates_df.describe()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(replicates_df['isometry_loss'], replicates_df['group_lasso_norm'])
ax.set_xlabel('isometry_loss')
ax.set_ylabel('group_lasso_norm')
plt.show()

In [ ]:
data_tensor = np.load('/Users/samsonkoelle/isometry-pursuit/data/ethanol_tensor.npy')  # (2, p, R)
R = data_tensor.shape[2]
losses = replicates_df['isometry_loss'].values
all_indices = replicates_df['basis_pursuit_indices'].tolist()

ncols = 5
nrows = (R + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3 * nrows))
axes = axes.flatten()

for i in range(R):
    ax = axes[i]
    cols = data_tensor[:, :, i]  # (2, p)
    selected = all_indices[i]
    mask = np.zeros(cols.shape[1], dtype=bool)
    mask[selected] = True

    ax.scatter(cols[0, ~mask], cols[1, ~mask], s=8, alpha=0.4, color='gray', label='other')
    ax.scatter(cols[0, mask], cols[1, mask], s=20, color='red', label='selected')
    ax.set_title(f'r{i}  loss={losses[i]:.2f}', fontsize=7)
    ax.set_xticks([])
    ax.set_yticks([])

for i in range(R, len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Jacobian columns in tangent space (red = selected by IP)', fontsize=11)
plt.tight_layout()
plt.show()